# 08-03 · 직접 구현한 회귀 검증 (statsmodels와 비교)

| 항목 | 내용 |
|---|---|
| 하는 일 | `02_회귀`의 `fe_ols`(직접 구현 고정효과 회귀)를 statsmodels의 더미변수 OLS와 비교 — ① 합성자료(참값 β=0.7), ② 실제 패널 일부 |
| 읽는 파일 | `02_회귀.py`(파일 경로로 불러옴), `outputs/07_분석/패널_기업분기.csv` |
| 만드는 파일 | 없음 (화면 출력만) |
| 인증키 | 필요 없음 (저장된 파일만 읽습니다) |
| 걸리는 시간 | 약 18초 — 2026-11-05, 이 컴퓨터에서 측정 |

> 이 노트북이 **원본**입니다. 같은 이름의 `.py`는 `scripts/노트북에서_py_만들기.py`가 여기서 자동으로 만든 파일이라 직접 고치지 않습니다. 원래 손으로 쓴 `.py`는 이 폴더의 `원본_스크립트/`에 보관했습니다.

분석 결과를 만드는 단계가 아니라 **계산 도구가 맞는지** 보는 단계입니다. 계수가 같고 군집 표준오차 비율이 기대와 맞으면 구현이 받아들여집니다.

### 준비 — 불러오기와 경로

필요한 도구를 불러오고, 이 폴더 위치(`HERE`)를 잡습니다. 맨 위 따옴표 글(독스트링)은 이 스크립트의 원래 사용설명서입니다.

In [1]:
"""직접 구현한 고정효과 회귀(02_회귀.py의 fe_ols)를 statsmodels(더미 변수 OLS)와 비교해 구현을 검증한다.
 1) 합성 자료(참값 β=0.7): 계수가 statsmodels와 같은지, 참값 근처인지
 2) 실제 패널의 주 표본 일부: M2 계수가 statsmodels와 같은지, 군집 표준오차 비율
실행: python -X utf8 03_회귀_구현검증.py
"""
import importlib.util
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
try:
    HERE = HERE      # .py로 실행할 때
except NameError:
    HERE = Path.cwd()                           # 노트북에서 실행할 때(노트북이 있는 폴더)

### 상수 — 입력·출력 경로와 규칙 값

In [2]:
spec = importlib.util.spec_from_file_location("reg", HERE / "02_회귀.py")
reg = importlib.util.module_from_spec(spec)
sys.path.insert(0, str(HERE.parent / "01_파일럿"))
spec.loader.exec_module(reg)
if hasattr(sys.stdout, "reconfigure"):          # 노트북의 출력 창에는 이 함수가 없다
    sys.stdout.reconfigure(encoding="utf-8")

#### `compare()`

In [3]:
def compare(df, y, x, fes, label):
    me = reg.fe_ols(df, y, [x], fes)
    d = df[list(dict.fromkeys([y, x] + fes + ["corp_code"]))].dropna()
    # 싱글톤 제거를 같은 방식으로
    while True:
        keep = np.ones(len(d), bool)
        for f in fes:
            keep &= d[f].map(d[f].value_counts()).values > 1
        if keep.all():
            break
        d = d[keep]
    form = f"{y} ~ {x} + " + " + ".join(f"C({f})" for f in fes)
    sm = smf.ols(form, data=d).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d["corp_code"])[0]})
    print(f"[{label}] 직접구현 β={me['beta'][0]:.6f} se={me['se'][0]:.5f} | statsmodels β={sm.params[x]:.6f} se={sm.bse[x]:.5f} | β 차이 {abs(me['beta'][0]-sm.params[x]):.2e}, se 비율 {me['se'][0]/sm.bse[x]:.3f} | N={me['N']}")
    return me, sm

#### `main()`

In [4]:
def main():
    rng = np.random.default_rng(1)
    F, T, I = 80, 14, 5
    firm = np.repeat(np.arange(F), T)
    t = np.tile(np.arange(T), F)
    ind = np.repeat(rng.integers(0, I, F), T)
    x = rng.normal(size=F * T) + 0.3 * (firm % 7)
    fe_f = rng.normal(size=F)[firm] * 2
    fe_it = rng.normal(size=I * T).reshape(I, T)[ind, t]
    y = 0.7 * x + fe_f + fe_it + rng.normal(size=F * T)
    d = pd.DataFrame({"y": y, "x": x, "corp_code": firm.astype(str), "ind_t": [f"{a}_{b}" for a, b in zip(ind, t)], "t": t.astype(str)})
    d = d.drop(d.sample(frac=0.08, random_state=2).index)     # 불균형 패널
    compare(d, "y", "x", ["corp_code", "ind_t"], "합성 M2 (참값 0.7)")
    compare(d, "y", "x", ["corp_code", "t"], "합성 M1 (참값 0.7)")
    P = pd.read_csv(HERE.parent.parent / "outputs" / "07_분석" / "패널_기업분기.csv", dtype={"corp_code": str})
    P["corp_code"] = P["corp_code"].str.zfill(8)
    P["ind_t"] = P["업종2"].astype(str) + "_" + P["t"].astype(str)
    P["Ex"] = P["E_pre_A"] * P["tight"]
    S = P[P["표본_주"]]
    firms = S["corp_code"].drop_duplicates().sample(150, random_state=3)
    compare(S[S["corp_code"].isin(firms)], "icr", "Ex", ["corp_code", "ind_t"], "실제 자료 150개 기업 부분표본 M2")

### (.py에만 들어가는 부분)

노트북에서는 `__name__`이 항상 `"__main__"`이라 아래 코드를 일반 셀로 두면 열 때마다 실행됩니다. 그래서 실행하지 않는 설명 셀에 두고, 변환 스크립트가 `.py`에만 넣습니다.

```python
if __name__ == "__main__":
    main()
```

In [5]:
main()

[합성 M2 (참값 0.7)] 직접구현 β=0.671051 se=0.03240 | statsmodels β=0.671051 se=0.03382 | β 차이 3.66e-15, se 비율 0.958 | N=1030
[합성 M1 (참값 0.7)] 직접구현 β=0.654483 se=0.04026 | statsmodels β=0.654483 se=0.04193 | β 차이 1.55e-15, se 비율 0.960 | N=1030


[실제 자료 150개 기업 부분표본 M2] 직접구현 β=-751.588426 se=965.45365 | statsmodels β=-751.588426 se=993.82062 | β 차이 5.91e-12, se 비율 0.971 | N=2889
